# Демо: функции, аргументы, область видимости

Прокликай Shift+Enter каждую ячейку и посмотри, как Python работает с функциями: объявление, вызов, параметры с дефолтами, переменное число аргументов, правила области видимости. В конце — три мини-задания.

## Часть 1. Объявление и вызов

Сейчас посмотрим, как Python отделяет объявление от вызова. `def` только запоминает функцию, тело не выполняется до первого вызова с круглыми скобками.

In [1]:
def percent(part, total):
    return part / total * 100

# Пока что Python только запомнил функцию — ничего не напечаталось.
print(percent)            # объект-функция, не результат вызова
print(percent(37, 1000))  # вот теперь функция отработала и вернула 3.7

<function percent at 0x1094b1310>
3.6999999999999997


А что если забыть `return`? Функция всё равно отработает, но молча вернёт `None` — Python не предупредит, и в дальнейшем коде будут странные ошибки вида «`NoneType` не поддерживает `+`».

In [2]:
def add_silently(a, b):
    a + b   # посчитали, но не вернули

result = add_silently(2, 3)
print(result)         # None
print(type(result))   # <class 'NoneType'>

None
<class 'NoneType'>


## Часть 2. Позиционные и именованные аргументы

При вызове аргументы передаются либо по порядку (позиционные), либо по имени (именованные). Именованные удобнее, когда у функции много флагов-параметров — читателю не надо помнить порядок.

In [3]:
def greet(name, greeting, punctuation):
    return f"{greeting}, {name}{punctuation}"

# Позиционные — порядок важен
print(greet("Аня", "Привет", "!"))

# Именованные — порядок не важен, читаемость лучше
print(greet(name="Аня", punctuation="!", greeting="Привет"))

# Смешать можно: сначала позиционные, потом именованные
print(greet("Аня", greeting="Привет", punctuation="!"))

Привет, Аня!
Привет, Аня!
Привет, Аня!


Обратный порядок (именованные перед позиционными) — `SyntaxError` ещё на этапе чтения файла. Покажем через `try/except`, чтобы ноутбук не упал:

In [4]:
# eval() позволяет нам поймать SyntaxError на этапе компиляции
try:
    eval('greet(name="Аня", "Привет", "!")')
except SyntaxError as e:
    print(f"SyntaxError: {e}")

SyntaxError: positional argument follows keyword argument (<string>, line 1)


## Часть 3. Значения по умолчанию

Параметру можно задать значение, которое подставится, если аргумент не передали. Параметры с дефолтами стоят **после** обычных.

In [5]:
def greet(name, greeting="Привет", punctuation="!"):
    return f"{greeting}, {name}{punctuation}"

print(greet("Аня"))                     # все дефолты
print(greet("Аня", "Здравствуй"))       # переопределили greeting
print(greet("Аня", punctuation="."))    # переопределили только punctuation

Привет, Аня!
Здравствуй, Аня!
Привет, Аня.


## Часть 4. Подводный камень: изменяемый объект как дефолт

Знаменитая ловушка. Если в качестве дефолта поставить `[]` или `{}`, Python создаёт его **один раз** при объявлении функции. Все вызовы без аргумента будут делить **тот же самый** список:

In [6]:
def add_item(item, basket=[]):
    basket.append(item)
    return basket

print(add_item("яблоко"))    # ['яблоко']
print(add_item("груша"))     # ['яблоко', 'груша']  — а должно быть только ['груша']!
print(add_item("банан"))     # ['яблоко', 'груша', 'банан']

['яблоко']
['яблоко', 'груша']
['яблоко', 'груша', 'банан']


Правильный шаблон — `None` как дефолт + создание нового объекта внутри:

In [7]:
def add_item(item, basket=None):
    if basket is None:
        basket = []
    basket.append(item)
    return basket

print(add_item("яблоко"))    # ['яблоко']
print(add_item("груша"))     # ['груша']  — теперь правильно
print(add_item("банан"))     # ['банан']

['яблоко']
['груша']
['банан']


## Часть 5. `*args` — переменное число позиционных аргументов

Звёздочка перед параметром говорит: «собери все оставшиеся позиционные аргументы в кортеж». Имя `args` — соглашение, можно назвать как угодно.

In [8]:
def total(*numbers):
    print(f"тип: {type(numbers).__name__}, значение: {numbers}")
    return sum(numbers)

print(total(1, 2, 3))           # numbers = (1, 2, 3)
print(total(10, 20, 30, 40))    # numbers = (10, 20, 30, 40)
print(total())                  # numbers = ()  — пустой кортеж

тип: tuple, значение: (1, 2, 3)
6
тип: tuple, значение: (10, 20, 30, 40)
100
тип: tuple, значение: ()
0


## Часть 6. `**kwargs` — переменное число именованных аргументов

Двойная звёздочка собирает оставшиеся именованные аргументы в словарь. Типичное применение — функции-обёртки, которые передают параметры дальше, не зная заранее их состав.

In [9]:
def log_event(**fields):
    print(f"тип: {type(fields).__name__}")
    for key, value in fields.items():
        print(f"  {key}: {value}")

log_event(user="alice", action="login", ip="10.0.0.1")

тип: dict
  user: alice
  action: login
  ip: 10.0.0.1


Полный шаблон сигнатуры — обычные параметры, потом `*args`, потом параметры с дефолтами, потом `**kwargs`. Любой другой порядок — `SyntaxError`.

In [10]:
def request(method, url, *headers, timeout=30, **params):
    print(f"method={method!r}, url={url!r}")
    print(f"headers={headers}")
    print(f"timeout={timeout}")
    print(f"params={params}")

request("GET", "/api/users",
        "Authorization: token", "Accept: json",
        timeout=60, page=2, limit=100)

method='GET', url='/api/users'
headers=('Authorization: token', 'Accept: json')
timeout=60
params={'page': 2, 'limit': 100}


## Часть 7. Распаковка коллекций при вызове

Те же `*` и `**` работают и при вызове: распаковывают список в позиционные аргументы, словарь — в именованные. Удобно, когда параметры лежат в коллекции.

In [11]:
def percent(part, total):
    return part / total * 100

values = [37, 1000]
print(percent(*values))         # 3.7  — то же что percent(37, 1000)

options = {"part": 12, "total": 1000}
print(percent(**options))       # 1.2  — то же что percent(part=12, total=1000)

3.6999999999999997
1.2


## Часть 8. `lambda` — анонимная функция в одну строку

`lambda` создаёт функцию-выражение без имени. Тело — одно выражение, результат которого автоматически возвращается. Главное применение — ключ сортировки или фильтр в `filter`/`map`.

In [12]:
square = lambda x: x ** 2
print(square(5))    # 25

# Сортировка списка пар по второму элементу
pairs = [(1, 5), (2, 3), (8, 1), (4, 7)]
print(sorted(pairs, key=lambda pair: pair[1]))

25
[(8, 1), (2, 3), (1, 5), (4, 7)]


## Часть 9. LEGB: правила области видимости

Python ищет имя в четырёх областях по порядку:

- **L**ocal — внутри текущей функции
- **E**nclosing — во внешних функциях (если функция вложена)
- **G**lobal — на уровне модуля
- **B**uilt-in — встроенные имена (`len`, `print`, ...)

Кто первый совпал — тот и победил. Покажем по очереди.

In [13]:
# Local выигрывает у Global
name = "global Аня"

def show():
    name = "local Боря"
    print(name)

show()         # local Боря — Local выигрывает
print(name)    # global Аня — снаружи без изменений

local Боря
global Аня


In [14]:
# Enclosing — переменная из внешней функции
def outer():
    message = "из outer"
    def inner():
        print(message)   # читаем из enclosing
    inner()

outer()

из outer


Читать переменную внешней области — можно. А вот **присвоить** ей значение из вложенной функции просто так нельзя — Python подумает, что это новая локальная переменная и поднимет `UnboundLocalError`:

In [15]:
def outer():
    counter = 0
    def inner():
        counter += 1   # пытаемся изменить enclosing — ошибка
    try:
        inner()
    except UnboundLocalError as e:
        print(f"UnboundLocalError: {e}")

outer()

UnboundLocalError: local variable 'counter' referenced before assignment


Чтобы разрешить запись — нужен `nonlocal` (для enclosing) или `global` (для модульного уровня). Подробнее про `nonlocal` — в демо про замыкания и декораторы.

## Мини-задания

Три коротких упражнения. Подсказок к именам и методам нет — вспомни сам.

**Задание 1.** Напиши функцию, которая принимает произвольное число чисел и возвращает их среднее. Если чисел нет — возвращай `0`.

**Задание 2.** Напиши функцию `merge_configs`, которая принимает один обязательный параметр `base_config` (словарь) и любое число именованных аргументов. Возвращает новый словарь — копию `base_config`, в которой значения переопределены именованными аргументами.

**Задание 3.** Что напечатает код ниже? Сначала угадай, потом запусти.

In [16]:
# Задание 1
# def mean(...):
#     ...

# Проверка:
# print(mean(1, 2, 3, 4, 5))   # 3.0
# print(mean())                # 0


In [17]:
# Задание 2
# base = {'lr': 0.01, 'epochs': 10, 'batch_size': 32}
# def merge_configs(...):
#     ...

# Проверка:
# print(merge_configs(base, lr=0.1))
# # {'lr': 0.1, 'epochs': 10, 'batch_size': 32}


In [18]:
# Задание 3 — твой прогноз для каждой строки впиши в комментарий:
x = 10

def f(value=x):
    return value

x = 99
print(f())   # ?


10
